# 02 - Data Cleaning
Basic cleaning steps: handle missing values, types, and duplicates.

In [ ]:
# ---------------------------------------------------------
# Notebook: 02_data_cleaning.ipynb
# Purpose: Clean and preprocess aggregated flight delay dataset for ML
# Author: Kushalgowda
# ---------------------------------------------------------

import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# ===============================
# 1️Load dataset
# ===============================
DATA_PATH = r'C:\Users\gowda\Downloads\FDS PROJECT\Airline_Flight_Delay_Prediction\dataset\flights_initial_snapshot.csv'
df = pd.read_csv(DATA_PATH)

print("Original shape:", df.shape)

# ===============================
# 2️Handle missing values
# ===============================

# Drop columns with more than 50% missing values
missing_threshold = 0.5
df = df.loc[:, df.isnull().mean() < missing_threshold]

# Fill missing numeric values with median, categorical with mode
for col in df.columns:
    if df[col].isnull().any():
        if pd.api.types.is_numeric_dtype(df[col]):
            df[col].fillna(df[col].median(), inplace=True)
        else:
            df[col].fillna(df[col].mode()[0], inplace=True)

# ===============================
# 3️Handle outliers (numeric only)
# ===============================
numeric_cols = df.select_dtypes(include=[np.number]).columns

for col in numeric_cols:
    q1 = df[col].quantile(0.01)
    q3 = df[col].quantile(0.99)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    df = df[df[col].between(lower, upper)]

print("🧹 Outliers removed (1st–99th percentile range filtering)")

# ===============================
# 4️Feature engineering
# ===============================
if {'arr_del15', 'arr_flights'}.issubset(df.columns):
    df['delay_rate'] = (df['arr_del15'] / df['arr_flights']).round(4)

if {'arr_cancelled', 'arr_flights'}.issubset(df.columns):
    df['cancel_rate'] = (df['arr_cancelled'] / df['arr_flights']).round(4)

if {'arr_diverted', 'arr_flights'}.issubset(df.columns):
    df['divert_rate'] = (df['arr_diverted'] / df['arr_flights']).round(4)

delay_cols = ['carrier_delay', 'weather_delay', 'nas_delay', 'security_delay', 'late_aircraft_delay']
if all(col in df.columns for col in delay_cols):
    df['total_delay_minutes'] = df[delay_cols].sum(axis=1)

# ===============================
# 6️Preserve original column order
# ===============================
df = df.reindex(sorted(df.columns), axis=1)

# ===============================
# 7️ Final sanity check
# ===============================
print("\n Cleaned shape:", df.shape)
print("Columns:", df.columns.tolist())
print(f" Missing values after cleaning: {df.isnull().sum().sum()}")
print(f" Memory usage: {df.memory_usage(deep=True).sum() / (1024**2):.2f} MB")

# ===============================
# 8️ Save cleaned dataset
# ===============================
SAVE_PATH = r'C:\Users\gowda\Downloads\FDS PROJECT\Airline_Flight_Delay_Prediction\dataset\flights_cleaned.csv'
df.to_csv(SAVE_PATH, index=False)
print(f"\n Cleaned dataset saved successfully to:\n{SAVE_PATH}")


✅ Original shape: (104289, 21)


C:\Users\gowda\AppData\Local\Temp\ipykernel_9392\1104294246.py:31: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df[col].fillna(df[col].median(), inplace=True)


🧹 Outliers removed (1st–99th percentile range filtering)

✅ Cleaned shape: (102904, 25)
📋 Columns: ['airport', 'airport_name', 'arr_cancelled', 'arr_del15', 'arr_delay', 'arr_diverted', 'arr_flights', 'cancel_rate', 'carrier', 'carrier_ct', 'carrier_delay', 'carrier_name', 'delay_rate', 'divert_rate', 'late_aircraft_ct', 'late_aircraft_delay', 'month', 'nas_ct', 'nas_delay', 'security_ct', 'security_delay', 'total_delay_minutes', 'weather_ct', 'weather_delay', 'year']
🧩 Missing values after cleaning: 0
💾 Memory usage: 43.01 MB

✅ Cleaned dataset saved successfully to:
C:\Users\gowda\Downloads\FDS PROJECT\Airline_Flight_Delay_Prediction\dataset\flights_cleaned.csv
